# microgpt + tiny-ton GPU inference

Run [Karpathy's microgpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95) — pure Python, zero dependencies, complete GPT training + inference.

Then replace the forward pass with **tiny-ton** JIT-compiled GPU kernels and benchmark.

**Before running:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

## 1. Download microgpt.py

In [ ]:
!wget -q https://gist.githubusercontent.com/karpathy/8627fe009c40f57531cb18360106ce95/raw/microgpt.py -O /content/microgpt.py
!wc -l /content/microgpt.py
!head -7 /content/microgpt.py

## 2. Run microgpt (CPU baseline)

Pure Python, no dependencies, no GPU. Trains a 1-layer GPT on character-level name generation.

`n_layer=1, n_embd=16, block_size=16, n_head=4, 1000 steps`.

In [ ]:
%%time
!cd /content && python3 microgpt.py

## 3. Install LLVM/MLIR 18 + build tiny-ton

In [ ]:
%%bash
set -e
echo '>>> Adding LLVM 18 apt repository...'
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
add-apt-repository -y 'deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main' > /dev/null 2>&1
echo '>>> Installing packages...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build > /dev/null 2>&1
pip install -q pybind11
echo '>>> Done. MLIR version:'
mlir-opt-18 --version 2>&1 | head -2

In [ ]:
%%bash
set -e
if [ ! -d /content/tiny-ton ]; then
  git clone https://github.com/ganeshnj/tiny-ton.git /content/tiny-ton
else
  echo 'tiny-ton already cloned, pulling latest...'
  cd /content/tiny-ton && git pull
fi

In [ ]:
%%bash
set -e
cd /content/tiny-ton
rm -rf build
cmake -G Ninja -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \
  -DTTN_ENABLE_CUDA=ON \
  -DTTN_ENABLE_PYTHON=ON \
  -DCUDAToolkit_ROOT=/usr/local/cuda \
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
echo '>>> Build complete.'

In [ ]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')
os.environ['TTN_SM_VERSION'] = 'sm_75'
import importlib
importlib.invalidate_caches()
import _tiny_ton_core as core
import tiny_ton as tt
print(f'has_cuda() = {core.has_cuda()}')
print(f'TTN_SM_VERSION = {os.environ["TTN_SM_VERSION"]}')

## 4. Train microgpt on CPU, extract weights

Run the original microgpt training loop (1000 steps), then extract trained weights as numpy arrays for GPU inference.

In [ ]:
import os, math, random
import numpy as np
random.seed(42)

if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')
    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))
    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

n_layer = 1; n_embd = 16; block_size = 16; n_head = 4
head_dim = n_embd // n_head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row]
print(f'num params: {len(params)}')

def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]
def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]
def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)
    for li in range(n_layer):
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]
    logits = linear(x, state_dict['lm_head'])
    return logits

learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m_buf = [0.0] * len(params)
v_buf = [0.0] * len(params)
num_steps = 1000

print(f'Training {num_steps} steps...')
for step in range(num_steps):
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)
    keys_t, values_t = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys_t, values_t)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses)
    loss.backward()
    lr_t = learning_rate * (1 - step / num_steps)
    for i, p in enumerate(params):
        m_buf[i] = beta1 * m_buf[i] + (1 - beta1) * p.grad
        v_buf[i] = beta2 * v_buf[i] + (1 - beta2) * p.grad ** 2
        m_hat = m_buf[i] / (1 - beta1 ** (step + 1))
        v_hat = v_buf[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0
    if (step + 1) % 100 == 0 or step == 0:
        print(f'step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}')

print('\nTraining complete. Extracting weights to numpy...')
def to_numpy(mat):
    return np.array([[v.data for v in row] for row in mat], dtype=np.float32)

weights_np = {}
for key, mat in state_dict.items():
    weights_np[key] = to_numpy(mat)
    print(f'  {key}: {weights_np[key].shape}')
print('Done.')

## 5. Define GPU forward pass

Replace `linear()`, `rmsnorm()`, `softmax()`, `relu()`, and multi-head attention with tiny-ton GPU kernels.

Same model architecture, same weights, just runs on GPU instead of Python loops.

In [ ]:
# --- linear / matvec kernel -------------------------------------------------

@tt.jit
def gpu_linear_kernel(W_ptr, x_ptr, y_ptr, in_features):
    pid = tt.program_id(0)
    tid = tt.arange(0, 64)
    mask = tid < in_features
    w = tt.load(W_ptr + pid * in_features + tid, mask=mask)
    x = tt.load(x_ptr + tid, mask=mask)
    dot = tt.reduce_sum(w * x)
    tt.store(y_ptr + pid, dot)

def gpu_linear(x, W_np):
    out_f, in_f = W_np.shape
    y = np.zeros(out_f, dtype=np.float32)
    gpu_linear_kernel[(out_f,)](W_np.flatten().copy(), x.copy(), y, in_f)
    return y

# --- rmsnorm (4 kernel launches) --------------------------------------------

@tt.jit
def rn_square(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, x * x, mask=mask)

@tt.jit
def rn_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def rn_rsqrt_mean(sum_ptr, n_ptr, out_ptr):
    tid = tt.arange(0, 64)
    s = tt.load(sum_ptr)
    n = tt.load(n_ptr)
    mean_eps = s / n + 1e-5
    scale = tt.rsqrt(mean_eps)
    tt.store(out_ptr, scale)

@tt.jit
def rn_mul_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x * s, mask=mask)

def gpu_rmsnorm(x, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_sq = np.zeros(N, dtype=np.float32)
    tmp_sum = np.zeros(1, dtype=np.float32)
    tmp_scl = np.zeros(1, dtype=np.float32)
    n_arr = np.array([float(N)], dtype=np.float32)
    out = np.zeros(N, dtype=np.float32)
    rn_square[grid](x, tmp_sq, N)
    rn_reduce_sum[(1,)](tmp_sq, tmp_sum, N)
    rn_rsqrt_mean[(1,)](tmp_sum, n_arr, tmp_scl)
    rn_mul_scalar[grid](x, tmp_scl, out, N)
    return out

# --- softmax (5 kernel launches) --------------------------------------------

@tt.jit
def sm_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

@tt.jit
def sm_sub_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x - s, mask=mask)

@tt.jit
def sm_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, tt.exp(x), mask=mask)

@tt.jit
def sm_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def sm_div_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x / s, mask=mask)

def gpu_softmax(x, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_max = np.zeros(1, dtype=np.float32)
    tmp_exp = np.zeros(N, dtype=np.float32)
    tmp_sum = np.zeros(1, dtype=np.float32)
    out = np.zeros(N, dtype=np.float32)
    sm_reduce_max[(1,)](x, tmp_max, N)
    sm_sub_scalar[grid](x, tmp_max, tmp_exp, N)
    sm_exp[grid](tmp_exp, tmp_exp, N)
    sm_reduce_sum[(1,)](tmp_exp, tmp_sum, N)
    sm_div_scalar[grid](tmp_exp, tmp_sum, out, N)
    return out

# --- relu -------------------------------------------------------------------

@tt.jit
def gpu_relu_kernel(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    y = tt.relu(x)
    tt.store(dst + off, y, mask=mask)

def gpu_relu(x, N):
    grid = (max(1, (N + 63) // 64),)
    out = np.zeros(N, dtype=np.float32)
    gpu_relu_kernel[grid](x, out, N)
    return out

# --- element-wise add kernel ------------------------------------------------

@tt.jit
def gpu_add_kernel(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(c_ptr + off, a + b, mask=mask)

def gpu_add(a, b, N):
    grid = (max(1, (N + 63) // 64),)
    out = np.zeros(N, dtype=np.float32)
    gpu_add_kernel[grid](a, b, out, N)
    return out

# --- GPU forward pass: gpt_gpu() -------------------------------------------

def gpt_gpu(token_id, pos_id, keys, values, w):
    tok_emb = w['wte'][token_id].copy()
    pos_emb = w['wpe'][pos_id].copy()
    x = gpu_add(tok_emb, pos_emb, n_embd)
    x = gpu_rmsnorm(x, n_embd)

    for li in range(n_layer):
        x_residual = x.copy()
        x = gpu_rmsnorm(x, n_embd)

        q = gpu_linear(x, w[f'layer{li}.attn_wq'])
        k = gpu_linear(x, w[f'layer{li}.attn_wk'])
        v = gpu_linear(x, w[f'layer{li}.attn_wv'])

        keys[li].append(k.copy())
        values[li].append(v.copy())

        x_attn = np.zeros(n_embd, dtype=np.float32)
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim].copy()

            seq_len = len(keys[li])
            K_h = np.array([ki[hs:hs+head_dim] for ki in keys[li]], dtype=np.float32)
            V_h = np.array([vi[hs:hs+head_dim] for vi in values[li]], dtype=np.float32)

            scores = np.zeros(seq_len, dtype=np.float32)
            gpu_linear_kernel[(seq_len,)](np.ascontiguousarray(K_h).flatten().copy(), q_h, scores, head_dim)

            sqrt_d = np.array([np.sqrt(float(head_dim))], dtype=np.float32)
            scores_scaled = np.zeros(seq_len, dtype=np.float32)
            sm_div_scalar[(1,)](scores, sqrt_d, scores_scaled, seq_len)

            attn_weights = gpu_softmax(scores_scaled, seq_len)

            V_h_T = np.ascontiguousarray(V_h.T)
            head_out = np.zeros(head_dim, dtype=np.float32)
            gpu_linear_kernel[(head_dim,)](V_h_T.flatten().copy(), attn_weights, head_out, seq_len)

            x_attn[hs:hs+head_dim] = head_out

        x = gpu_linear(x_attn, w[f'layer{li}.attn_wo'])
        x = gpu_add(x, x_residual, n_embd)

        x_residual = x.copy()
        x = gpu_rmsnorm(x, n_embd)
        x = gpu_linear(x, w[f'layer{li}.mlp_fc1'])
        x = gpu_relu(x, 4 * n_embd)
        x = gpu_linear(x, w[f'layer{li}.mlp_fc2'])
        x = gpu_add(x, x_residual, n_embd)

    logits = gpu_linear(x, w['lm_head'])
    return logits

print('GPU forward pass defined.')

## 6. GPU inference — generate names

Same inference loop as microgpt but using GPU kernels for the forward pass.

In [ ]:
temperature = 0.5
random.seed(42)
np.random.seed(42)

print('--- GPU inference (tiny-ton kernels) ---')
for sample_idx in range(20):
    keys_inf = [[] for _ in range(n_layer)]
    values_inf = [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt_gpu(token_id, pos_id, keys_inf, values_inf, weights_np)
        scaled = (logits / temperature).astype(np.float32)
        probs = gpu_softmax(scaled, vocab_size)
        probs = np.clip(probs, 0, None)
        probs = probs / probs.sum()
        token_id = np.random.choice(vocab_size, p=probs)
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f'sample {sample_idx+1:2d}: {"" .join(sample)}')

## 7. Benchmark — CPU vs GPU inference

Time both CPU and GPU inference for 20 samples.

**Note:** at n_embd=16, GPU will likely be slower due to kernel launch overhead (~55 launches per forward pass for tiny vectors). Real speedups come with fusion (Stage 3) and larger models.

In [ ]:
import time

def cpu_inference(n_samples=20):
    """Original microgpt inference using Value-based forward pass."""
    temperature = 0.5
    random.seed(123)
    samples = []
    for _ in range(n_samples):
        keys_inf = [[] for _ in range(n_layer)]
        values_inf = [[] for _ in range(n_layer)]
        token_id = BOS
        sample = []
        for pos_id in range(block_size):
            logits = gpt(token_id, pos_id, keys_inf, values_inf)
            scaled = [Value(l.data / temperature) for l in logits]
            probs = softmax(scaled)
            token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if token_id == BOS:
                break
            sample.append(uchars[token_id])
        samples.append(''.join(sample))
    return samples

def gpu_inference(n_samples=20):
    """GPU inference using tiny-ton kernels."""
    temperature = 0.5
    random.seed(123)
    np.random.seed(123)
    samples = []
    for _ in range(n_samples):
        keys_inf = [[] for _ in range(n_layer)]
        values_inf = [[] for _ in range(n_layer)]
        token_id = BOS
        sample = []
        for pos_id in range(block_size):
            logits = gpt_gpu(token_id, pos_id, keys_inf, values_inf, weights_np)
            scaled = (logits / temperature).astype(np.float32)
            probs = gpu_softmax(scaled, vocab_size)
            probs = np.clip(probs, 0, None)
            probs = probs / probs.sum()
            token_id = np.random.choice(vocab_size, p=probs)
            if token_id == BOS:
                break
            sample.append(uchars[token_id])
        samples.append(''.join(sample))
    return samples

# Warmup GPU (first launch compiles kernels)
print('Warming up GPU kernels...')
_ = gpu_inference(1)

N_SAMPLES = 20

t0 = time.perf_counter()
cpu_samples = cpu_inference(N_SAMPLES)
cpu_time = time.perf_counter() - t0

t0 = time.perf_counter()
gpu_samples = gpu_inference(N_SAMPLES)
gpu_time = time.perf_counter() - t0

print(f'\n{"="*60}')
print(f'  CPU inference ({N_SAMPLES} samples): {cpu_time:.3f}s')
print(f'  GPU inference ({N_SAMPLES} samples): {gpu_time:.3f}s')
print(f'  Ratio: {cpu_time/gpu_time:.2f}x {"faster" if gpu_time < cpu_time else "slower"} on GPU')
print(f'{"="*60}')

print(f'\nCPU samples: {cpu_samples[:5]}')
print(f'GPU samples: {gpu_samples[:5]}')